# Flickr30k Paper Tables

Notebook xuất các bảng CSV và Markdown phục vụ báo cáo/bài báo. Mọi metric
BLEU, METEOR, ROUGE-L và CIDEr-D được nhân 100, làm tròn hai chữ số.
Notebook chỉ đọc kết quả đã khóa và có thể chạy song song với
`F30K_Final_Closeout`.

## Kaggle input bắt buộc — đúng 6 output

1. `F30K_Global_Val_Lock`;
2. `F30K_DenseMap_Closeout`;
3. `F30K_Val_Report`;
4. `F30K_T3S3_HeldOut_Test_OneTime`;
5. `F30K_T3S3_Deployment_Lock`;
6. `F30K_T3S3_CPU_Proxy`.

Runtime: CPU, Accelerator None, Internet Off; dự kiến dưới 2 phút.


## Cell 1 — Xuất bảng kết quả khoa học và triển khai


In [1]:
import csv
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
EXPERIMENT_ID = "F30K_Paper_Tables"
WORK_ROOT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT", f"/kaggle/working/{EXPERIMENT_ID}"
))
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def now_utc():
    return datetime.now(timezone.utc).isoformat()

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def unique_named(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (f"Cần đúng một {name}", matches)
    return matches[0]

def load(name):
    path = unique_named(name)
    return path, json.loads(path.read_text(encoding="utf-8"))

def write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

def verify_manifest(root, manifest):
    rows = manifest["artifacts"]
    for row in rows:
        path = root / row["relative_path"]
        assert path.is_file(), path
        assert path.stat().st_size == int(row["size_bytes"]), path
        assert sha256(path) == row["sha256"], path
    return len(rows)


_, global_lock = load("F30K_GLOBAL_VALIDATION_LOCK.json")
_, dense = load("F30K_DenseMap_Closeout_RESULT.json")
_, val_report = load("F30K_Val_Report_RESULT.json")
_, heldout = load("F30K_T3S3_HeldOut_Test_OneTime_RESULT.json")
_, deployment = load("F30K_T3S3_Deployment_Lock_RESULT.json")
_, cpu_proxy = load("F30K_T3S3_CPU_Proxy_RESULT.json")

assert global_lock["status"] == "locked"
assert dense["decision"] == "STOP_VALIDATION_NO_GAIN"
assert val_report["held_out_test_read"] is False
assert heldout["held_out_test_authorization_remaining"] == 0
assert deployment["decision"]["unconditional_production_release"] == "not authorized"
assert "not an Android device benchmark" in cpu_proxy["role"]

method_columns = [
    "pair_id", "method", "baseline_CIDEr_D_mean_x100",
    "kd_CIDEr_D_mean_x100", "delta_CIDEr_D_mean_x100",
    "combined_ci95_lower_x100", "combined_ci95_upper_x100", "decision",
]
method_path = WORK_ROOT / "table_1_validation_methods_x100.csv"
with method_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=method_columns)
    writer.writeheader()
    for source in global_lock["candidate_registry"]:
        row = dict(source)
        for key in method_columns:
            if key.endswith("_x100") and isinstance(row.get(key), (int, float)):
                row[key] = f"{row[key]:.2f}"
        writer.writerow({key: row[key] for key in method_columns})

test_metrics = heldout["metrics_x100_rounded_2dp"]
test_path = WORK_ROOT / "table_2_heldout_test_x100.csv"
with test_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow([
        "model", "seed", "BLEU_1_x100", "BLEU_4_x100",
        "METEOR_x100", "ROUGE_L_x100", "CIDEr_D_x100", "CE_loss",
    ])
    writer.writerow([
        "T3S3 Direct Logit-KD", heldout["deployment_seed"],
        f"{test_metrics['BLEU_1']:.2f}", f"{test_metrics['BLEU_4']:.2f}",
        f"{test_metrics['METEOR']:.2f}", f"{test_metrics['ROUGE_L']:.2f}",
        f"{test_metrics['CIDEr_D']:.2f}",
        f"{heldout['metrics_raw']['ce_loss']:.6f}",
    ])

deploy_path = WORK_ROOT / "table_3_deployment.csv"
with deploy_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(["indicator", "value", "interpretation"])
    writer.writerows([
        ["FP32 reference size", f"{deployment['size']['fp32_reference_MiB']:.2f} MiB", "encoder + decoder"],
        ["Selected mixed package", f"{deployment['size']['selected_MiB']:.2f} MiB", "encoder FP32 + decoder dynamic INT8"],
        ["Size reduction", f"{deployment['size']['reduction_fraction'] * 100:.2f}%", "relative to FP32"],
        ["INT8 CIDEr-D delta ×100", f"{deployment['noninferiority']['observed_delta_CIDEr_D'] * 100:+.2f}", "Validation"],
        ["INT8 CI95 ×100", f"[{deployment['noninferiority']['ci95_lower'] * 100:.2f}; {deployment['noninferiority']['ci95_upper'] * 100:.2f}]", "boundary-sensitive"],
        ["Exact caption agreement", f"{0.7564102564102564 * 100:.2f}%", "below 90% strict gate"],
        ["CPU proxy 1-thread E2E", f"{cpu_proxy['summaries']['1']['end_to_end']['mean_ms']:.2f} ms", "not Android"],
        ["CPU proxy 4-thread E2E", f"{cpu_proxy['summaries']['4']['end_to_end']['mean_ms']:.2f} ms", "not Android"],
    ])

markdown_path = WORK_ROOT / "F30K_PAPER_TABLES.md"
lines = [
    "# Flickr30k — bảng kết quả đã khóa",
    "",
    "## So sánh phương pháp trên Validation",
    "",
    "| Cặp | Phương pháp | Baseline CIDEr-D | KD CIDEr-D | Δ | CI 95% | Quyết định |",
    "|---|---|---:|---:|---:|---:|---|",
]
for row in global_lock["candidate_registry"]:
    lines.append(
        f"| {row['pair_id']} | {row['method']} | "
        f"{row['baseline_CIDEr_D_mean_x100']:.2f} | "
        f"{row['kd_CIDEr_D_mean_x100']:.2f} | "
        f"{row['delta_CIDEr_D_mean_x100']:+.2f} | "
        f"[{row['combined_ci95_lower_x100']:.2f}; "
        f"{row['combined_ci95_upper_x100']:.2f}] | "
        f"{row['decision']} |"
    )
lines += [
    "",
    "## Held-out Test",
    "",
    "| BLEU-1 | BLEU-4 | METEOR | ROUGE-L | CIDEr-D | CE |",
    "|---:|---:|---:|---:|---:|---:|",
    f"| {test_metrics['BLEU_1']:.2f} | {test_metrics['BLEU_4']:.2f} | "
    f"{test_metrics['METEOR']:.2f} | {test_metrics['ROUGE_L']:.2f} | "
    f"{test_metrics['CIDEr_D']:.2f} | {heldout['metrics_raw']['ce_loss']:.6f} |",
    "",
    "Các metric chất lượng đã nhân 100 và làm tròn hai chữ số; CE giữ thang gốc.",
    "Held-out Test đã mở đúng một lần và không còn quyền chạy lại.",
]
markdown_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
artifact_paths = [method_path, test_path, deploy_path, markdown_path]
result = {
    "schema_version": "f30k_paper_tables_v1",
    "created_at_utc": now_utc(),
    "status": "complete",
    "experiment_id": EXPERIMENT_ID,
    "metric_display_scale": 100,
    "metric_rounding_decimals": 2,
    "held_out_test_reopened": False,
    "artifacts": [
        {
            "relative_path": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in artifact_paths
    ],
}
write_json(result_path, result)
manifest_path = WORK_ROOT / "MANIFEST.json"
all_artifacts = artifact_paths + [result_path]
write_json(manifest_path, {
    "schema_version": "f30k_paper_tables_manifest_v1",
    "created_at_utc": now_utc(),
    "artifact_count": len(all_artifacts),
    "artifacts": [
        {
            "relative_path": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in all_artifacts
    ],
})
verify_manifest(WORK_ROOT, json.loads(manifest_path.read_text(encoding="utf-8")))

print("Status                 : COMPLETE")
print("Metric display         : ×100 / 2 decimals")
print("Held-out Test reopened : False")
print("Validation table       :", method_path)
print("Held-out table         :", test_path)
print("Deployment table       :", deploy_path)
print("Markdown               :", markdown_path)
print("Manifest               :", manifest_path)


Status                 : COMPLETE
Metric display         : ×100 / 2 decimals
Held-out Test reopened : False
Validation table       : /kaggle/working/F30K_Paper_Tables/table_1_validation_methods_x100.csv
Held-out table         : /kaggle/working/F30K_Paper_Tables/table_2_heldout_test_x100.csv
Deployment table       : /kaggle/working/F30K_Paper_Tables/table_3_deployment.csv
Markdown               : /kaggle/working/F30K_Paper_Tables/F30K_PAPER_TABLES.md
Manifest               : /kaggle/working/F30K_Paper_Tables/MANIFEST.json
